# House Prices — V6: Equal Blend of Two Prediction Pipelines

**Recorded Kaggle log RMSE** 0.11419

This notebook recreates the scored submission from saved model predictions and the repository data. Run the cells in order from a Python kernel with the packages in [final_versions/requirements.txt](final_versions/requirements.txt) installed.

## 1. Load the competition data

In [1]:
from io import BytesIO
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd()
root = next(
    (path for path in (current, *current.parents)
     if (path / "final_versions" / "rebuild.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project repository.")
sys.path.insert(0, str(root))
from final_versions import rebuild

data = root / "Dataset"
if not data.is_dir():
    data = root / "house-prices-advanced-regression-techniques"
test = pd.read_csv(data / "test.csv", keep_default_na=False, na_values=["", "NA"])
assert len(test) == 1459 and test["Id"].is_unique

version = "V6"

## 2. Build the predictions

V6 combines V5 with a second prediction pipeline built from support vector
regression, CatBoost and a residual adjustment. The two predictions
receive equal weight on the log-price scale.

In [2]:
v5 = (rebuild.SUBMISSIONS / "v5.csv").read_bytes()
payload = rebuild.rebuild_v6(test["Id"], v5)
experiment = pd.read_csv(rebuild.COMPONENTS / "submission_experiment_01.csv")
display(pd.DataFrame({
    "Id": test["Id"],
    "V5 estimate": rebuild.read_prices(v5),
    "Second estimate": experiment["SalePrice"],
    "V6 estimate": rebuild.read_prices(payload),
}).head(5).round(2))

,Id,V5 estimate,Second estimate,V6 estimate
0,1461,120895.14,122621.61,121755.31
1,1462,165989.72,161442.14,163700.14
2,1463,185296.29,185143.80,185220.03
3,1464,196017.04,198076.62,197044.14
4,1465,189213.13,188963.27,189088.16


## 3. Verify and save the submission

In [3]:
rebuild.compare_with_submission(version, payload, test["Id"])
output = root / "rebuilt" / rebuild.VERSION_FILES[version]
output.parent.mkdir(parents=True, exist_ok=True)
output.write_bytes(payload)
submission = pd.read_csv(output)
scored = rebuild.SUBMISSIONS / output.name
display(pd.DataFrame([{
    "Output": str(output.relative_to(root)),
    "Rows": len(submission),
    "Comparison with scored CSV": (
        "Byte-identical" if payload == scored.read_bytes() else "Numerically matched"
    ),
}]))
display(submission.head(5).round(2))

,Output,Rows,Comparison with scored CSV
0,rebuilt/v6.csv,1459,Byte-identical


,Id,SalePrice
0,1461,121755.31
1,1462,163700.14
2,1463,185220.03
3,1464,197044.14
4,1465,189088.16


## Results

The two components scored **0.11599** and **0.11646** on Kaggle.
Their equal blend scored **0.11419**.